In [ ]:
# ==================== PART 1: INSTALL LIBRARIES ====================
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torchaudio
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries installed successfully!\n")


In [ ]:
# ==================== PART 2: LOAD DATA & LEAK-FREE SPLIT ====================
import os, sys, json as _json
print("="*60)
print("📥 LOADING DATASET WITH LEAK-FREE SPLIT...")
print("="*60)

# Load dataset
dataset = load_dataset("hustep-lab/ViSEC", trust_remote_code=True)
df = dataset['train'].to_pandas()

print(f"✓ Original dataset size: {len(df)} samples")
print(f"Unique emotions: {df['emotion'].unique()}")
print(f"\nEmotion distribution:")
print(df['emotion'].value_counts())

# Keep only path and emotion
df_clean = df[['path', 'emotion']].copy()

# Encode emotion labels
le = LabelEncoder()
df_clean['label'] = le.fit_transform(df_clean['emotion'])
emotion_labels = le.classes_
num_labels = len(emotion_labels)

print(f"\nEmotion mapping:")
for idx, emotion in enumerate(emotion_labels):
    print(f"  {idx}: {emotion}")

# Read fixed split_manifest.json as the single source of truth
manifest_path = os.path.join(os.path.dirname(os.path.abspath(".")), "split_manifest.json")
if not os.path.exists(manifest_path):
    manifest_path = "../split_manifest.json"
if not os.path.exists(manifest_path):
    manifest_path = "split_manifest.json"

print(f"\nLoading manifest from: {manifest_path}")
with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest = _json.load(f)

train_idx = manifest['train_indices']
val_idx = manifest['val_indices']
test_idx = manifest['test_indices']

X_train = df_clean['path'].iloc[train_idx].values
y_train = df_clean['label'].iloc[train_idx].values
X_val = df_clean['path'].iloc[val_idx].values
y_val = df_clean['label'].iloc[val_idx].values
X_test = df_clean['path'].iloc[test_idx].values
y_test = df_clean['label'].iloc[test_idx].values

print(f"\n✓ Train: {len(X_train)} samples")
print(f"✓ Val:   {len(X_val)} samples")
print(f"✓ Test:  {len(X_test)} samples")


In [ ]:
# ==================== PART 3: UTILITY FUNCTIONS ====================
def load_audio(path_dict, sr=16000):
    """Load audio file from bytes or path"""
    try:
        import io
        if isinstance(path_dict, dict) and 'bytes' in path_dict:
            audio_bytes = path_dict['bytes']
            audio, _ = librosa.load(io.BytesIO(audio_bytes), sr=sr)
            return audio
        elif isinstance(path_dict, str):
            audio, _ = librosa.load(path_dict, sr=sr)
            return audio
        elif isinstance(path_dict, dict) and 'path' in path_dict:
            audio, _ = librosa.load(path_dict['path'], sr=sr)
            return audio
    except Exception as e:
        return None
    return None

def plot_confusion_matrix(y_true, y_pred, labels, title):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(title, fontsize=16, pad=20)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()
    return cm


In [ ]:
# ==================== PART 4: DEFINE ECAPA-TDNN MODEL ====================
print(f"\n{'='*60}")
print("🎵 DEFINE ECAPA-TDNN ARCHITECTURE")
print(f"{'='*60}")

class ECAPA_TDNN(nn.Module):
    """ECAPA-TDNN for speech embeddings"""
    def __init__(self, input_size=80, channels=256, emb_size=192):
        super(ECAPA_TDNN, self).__init__()

        self.conv1 = nn.Conv1d(input_size, channels, 5, padding=2)
        self.bn1 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

        self.layer1 = nn.Sequential(
            nn.Conv1d(channels, channels, 3, padding=1, dilation=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.layer2 = nn.Sequential(
            nn.Conv1d(channels, channels, 3, padding=2, dilation=2),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.layer3 = nn.Sequential(
            nn.Conv1d(channels, channels, 3, padding=3, dilation=3),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.conv2 = nn.Conv1d(channels * 3, channels, 1)
        self.bn2 = nn.BatchNorm1d(channels)

        self.fc1 = nn.Linear(channels * 2, channels)
        self.bn3 = nn.BatchNorm1d(channels)

        self.fc2 = nn.Linear(channels, emb_size)
        self.bn4 = nn.BatchNorm1d(emb_size)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)

        x = torch.cat([x1, x2, x3], dim=1)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)

        mean = x.mean(dim=2)
        std = x.std(dim=2)
        x = torch.cat([mean, std], dim=1)

        x = self.fc1(x)
        x = self.bn3(x)
        x = self.relu(x)

        x = self.fc2(x)
        x = self.bn4(x)

        return x

print("✓ ECAPA-TDNN architecture defined!")


In [ ]:
# ==================== PART 5: FEATURE EXTRACTION ====================
print(f"\n{'='*60}")
print("🔊 EXTRACT MEL-SPECTROGRAM FEATURES")
print(f"{'='*60}")

def extract_features(audio, sr=16000, n_mels=80):
    """Extract mel-spectrogram from audio"""
    mel_spec = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels,
        n_fft=512, hop_length=160, win_length=400
    )
    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    log_mel_spec = (log_mel_spec - log_mel_spec.mean()) / (log_mel_spec.std() + 1e-8)
    return log_mel_spec

def prepare_features(audio_paths, labels, split_name=""):
    """Prepare features for a given split"""
    features_list = []
    labels_list = []
    for i, (path, label) in enumerate(zip(audio_paths, labels)):
        if i % 500 == 0:
            print(f"  [{split_name}] Processed {i}/{len(audio_paths)} samples...")
        audio = load_audio(path, sr=16000)
        if audio is not None and len(audio) > 0:
            features = extract_features(audio)
            features_list.append(features)
            labels_list.append(label)
    return features_list, np.array(labels_list)

# Extract features for all splits
print("\nExtracting features for train data...")
X_train_feat, y_train_clean = prepare_features(X_train, y_train, "Train")

print("\nExtracting features for val data...")
X_val_feat, y_val_clean = prepare_features(X_val, y_val, "Val")

print("\nExtracting features for test data...")
X_test_feat, y_test_clean = prepare_features(X_test, y_test, "Test")

print(f"\n✓ Train features: {len(X_train_feat)} samples")
print(f"✓ Val features: {len(X_val_feat)} samples")
print(f"✓ Test features: {len(X_test_feat)} samples")


In [ ]:
# ==================== PART 6: CREATE DATASET AND DATALOADER ====================
print(f"\n{'='*60}")
print("🔧 CREATE PYTORCH DATASET")
print(f"{'='*60}")

from torch.utils.data import Dataset, DataLoader

class AudioFeaturesDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feat = torch.FloatTensor(self.features[idx])
        label = torch.LongTensor([self.labels[idx]])
        return feat, label.squeeze()

def collate_fn(batch):
    """Custom collate function to pad sequences"""
    features, labels = zip(*batch)
    max_len = max([f.shape[1] for f in features])
    padded_features = []
    for feat in features:
        if feat.shape[1] < max_len:
            pad_len = max_len - feat.shape[1]
            feat = torch.nn.functional.pad(feat, (0, pad_len))
        padded_features.append(feat)
    features = torch.stack(padded_features)
    labels = torch.stack(list(labels))
    return features, labels

# Create datasets
train_dataset = AudioFeaturesDataset(X_train_feat, y_train_clean)
val_dataset = AudioFeaturesDataset(X_val_feat, y_val_clean)
test_dataset = AudioFeaturesDataset(X_test_feat, y_test_clean)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")
print(f"✓ Test batches: {len(test_loader)}")


In [ ]:
# ==================== PART 7: TRAIN MODEL ====================
print(f"\n{'='*60}")
print("🤖 TRAIN ECAPA-TDNN MODEL")
print(f"{'='*60}")

class EmotionClassifier(nn.Module):
    def __init__(self, num_classes):
        super(EmotionClassifier, self).__init__()
        self.ecapa = ECAPA_TDNN(input_size=80, channels=256, emb_size=192)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(192, num_classes)

    def forward(self, x):
        emb = self.ecapa(x)
        emb = self.dropout(emb)
        out = self.classifier(emb)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = EmotionClassifier(num_labels).to(device)
print(f"\n✓ Model parameters: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003, weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

warmup_epochs = 2

def train_epoch(model, loader, criterion, optimizer, device, epoch=0):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    # Learning rate warmup — only override during warmup phase
    if epoch < warmup_epochs:
        lr_scale = (epoch + 1) / warmup_epochs
        for param_group in optimizer.param_groups:
            param_group['lr'] = 0.0003 * lr_scale

    for batch_idx, (features, labels) in enumerate(loader):
        features, labels = features.to(device), labels.to(device)
        if torch.isnan(features).any():
            continue
        optimizer.zero_grad()
        outputs = model(features)
        if torch.isnan(outputs).any():
            continue
        loss = criterion(outputs, labels)
        if torch.isnan(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total

def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device)
            outputs = model(features)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

# Training loop — uses VALIDATION set for early stopping (leak-free)
print("\n🚀 Starting training...")
num_epochs = 100
best_val_f1 = 0
patience = 20
patience_counter = 0

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, epoch)

    if np.isnan(train_loss):
        print(f"⚠️ Training loss is NaN at epoch {epoch+1}, stopping training")
        break

    # Evaluate on VALIDATION set (never on test during training)
    val_preds, val_labels_eval = evaluate(model, val_loader, device)
    val_acc = accuracy_score(val_labels_eval, val_preds)
    val_f1 = f1_score(val_labels_eval, val_preds, average='weighted')

    # Learning rate scheduling (skip warmup epochs)
    if epoch >= warmup_epochs:
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_f1)
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr < old_lr:
            print(f"  ⚠️ Learning rate reduced: {old_lr:.6f} → {new_lr:.6f}")

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{num_epochs} [LR: {current_lr:.6f}]:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")

    # Save best model based on VALIDATION F1
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_ecapa_model.pth')
        print(f"  ✓ Saved best model with Val F1: {best_val_f1:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⚠️ Early stopping triggered after {epoch+1} epochs")
            break

print("\n✓ Training completed!")


In [ ]:
# ==================== PART 8: FINAL TEST EVALUATION (exactly once) ====================
print(f"\n{'='*60}")
print("📊 FINAL TEST EVALUATION (leak-free)")
print(f"{'='*60}")

# Load best model (selected on validation set)
model.load_state_dict(torch.load('best_ecapa_model.pth', weights_only=True))

print("\nEvaluating on test set (exactly once)...")
start_time = time.time()
test_preds, test_labels_eval = evaluate(model, test_loader, device)
inference_time = time.time() - start_time

ecapa_f1 = f1_score(test_labels_eval, test_preds, average='weighted')
ecapa_acc = accuracy_score(test_labels_eval, test_preds)
ecapa_latency = (inference_time / len(test_labels_eval)) * 1000

print(f"\n{'='*60}")
print("📈 FINAL RESULTS")
print(f"{'='*60}")
print(f"✓ Accuracy: {ecapa_acc:.4f}")
print(f"✓ F1 Score (weighted): {ecapa_f1:.4f}")
print(f"✓ Best Val F1 (training): {best_val_f1:.4f}")
print(f"✓ Latency: {ecapa_latency:.2f} ms/sample")
print(f"✓ Total inference time: {inference_time:.2f}s")

print("\nClassification Report:")
print(classification_report(test_labels_eval, test_preds, target_names=emotion_labels))

plot_confusion_matrix(test_labels_eval, test_preds, emotion_labels,
                     "ECAPA-TDNN Confusion Matrix")


In [ ]:
# ==================== PART 9: DETAILED ANALYSIS ====================
print(f"\n{'='*60}")
print("📊 DETAILED ANALYSIS")
print(f"{'='*60}")

from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1_per_class, support = precision_recall_fscore_support(
    test_labels_eval, test_preds, average=None
)

results_df = pd.DataFrame({
    'Emotion': emotion_labels,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1_per_class,
    'Support': support
})

print("\nResults per emotion:")
print(results_df.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.bar(emotion_labels, f1_per_class, color='coral', edgecolor='darkred', alpha=0.7)
plt.xlabel('Emotion', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.title('F1-Score per Emotion (ECAPA-TDNN)', fontsize=14, pad=20)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)

for i, v in enumerate(f1_per_class):
    plt.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n✅ ECAPA-TDNN MODEL EVALUATION COMPLETED!")
